# Corpus audit and article embeddings
Mary Kate Carara — modern pipeline. Run from the project uv environment.

The proposal reports 11 attributes and coverage through April 2020; the supplied subset has 15 columns and ends March 26, 2020. Keep all 7,337 records and flag duplicate bodies. The regex-filtered corpus describes selected news coverage, not public opinion.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'projectproposal.tex').exists())
ROOT = REPO/'modern-methods/project'
sys.path.insert(0, str(ROOT))
from src import topic_helpers as h
OUT = ROOT/'Results/modern'


In [2]:
df, summary = h.audit_corpus(ROOT)
summary

{'rows': 7337,
 'source_columns': 15,
 'publications': 25,
 'date_min': '2016-01-01',
 'date_max': '2020-03-26',
 'duplicate_bodies': 5,
 'missing': {'article_id': 0,
  'X1': 0,
  'Unnamed: 0': 0,
  'date': 0,
  'year': 0,
  'month': 0,
  'day': 0,
  'author': 1381,
  'title': 0,
  'article': 0,
  'url': 0,
  'section': 3668,
  'publication': 0,
  'title_match': 0,
  'body_hits': 0,
  'word_count': 0,
  'text': 0,
  'duplicate_body': 0,
  'measured_word_count': 0,
  'word_count_difference': 0},
 'word_count_mismatches': 0,
 'word_quantiles': {0.0: 150.0,
  0.25: 330.0,
  0.5: 496.0,
  0.75: 819.0,
  0.9: 1283.0,
  0.99: 5572.560000000042,
  1.0: 28048.0}}

## Inspect the corpus
Read `Results/modern/corpus_review.csv` and fill relevance, boilerplate, malformed, and notes. These are audit flags, not automatic exclusions.

In [3]:
display(df.sample(10, random_state=42)[['article_id','title','publication','text']])
display(df[['duplicate_body','word_count_difference']].value_counts().head(10))

,article_id,title,publication,text
996,294717,Udacity open sources its self-driving car simu...,TechCrunch,Self-driving cars require self-driving car sof...
736,221300,RPT-U.S. regulators could waive some safety ru...,Reuters,(Repeats to additional subscribers with no cha...
1244,334550,Fiat Chrysler may add more self-driving suppli...,Reuters,MILAN/DETROIT (Reuters) - Fiat Chrysler Automo...
1181,321565,Tech's terrible week,Axios,"From a fatal car crash to a data nightmare, tu..."
3393,786573,Apple ramps up self-driving program as Waymo s...,Reuters,SAN FRANCISCO (Reuters) - Apple Inc has sharpl...
2303,508115,Waymo's self-driving cars improve performance ...,Reuters,(Reuters) - Alphabet Inc’s Waymo said its flee...
230,56838,Driverless cars can operate in California as e...,Vox,Driverless cars will begin operating on Califo...
3207,684367,Tesla's Autopilot couldn't stand up to these h...,Mashable,A driver of a Tesla Model S died in early May ...
1740,422047,Innoviz extends funding round to $170 million ...,TechCrunch,"Just a few months ago, Innoviz became one of t..."
6731,2375414,Apple Wozniak given up believing in self-drivi...,CNBC,Steve Wozniak helped launch the personal compu...


duplicate_body  word_count_difference
False           0                        7332
True            0                           5
Name: count, dtype: int64

## Full-text contextual embeddings
The lecture distinguishes contextual sentence embeddings from counts, static word2vec, and arbitrary raw BERT outputs. This notebook uses the chapter’s GTE-small encoder, with 480-token chunks and token-weighted pooling. Every article still contributes one normalized vector. Pooling can blur multiple themes.

Model downloads require network access. Cache manifests pin the resolved revision; interrupted embedding batches resume at article boundaries.

In [4]:
pilot = df.sample(200, random_state=42).sort_index()
pilot_embeddings, pilot_cache = h.embed_corpus(pilot, ROOT, pilot=True)
display(pd.read_csv(pilot_cache/'chunk_diagnostics.csv').describe())

/Users/marykate/Desktop/F26/DataMiningProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11971.18it/s]


Embedded 200/200 articles


,article_id,chunks,tokens,largest_chunk
count,2.000000e+02,200.000000,200.000000,200.000000
mean,1.061271e+06,2.395000,908.675000,448.395000
std,7.872619e+05,1.984608,928.366967,61.610863
min,1.134800e+04,1.000000,222.000000,222.000000
25%,4.181972e+05,1.000000,456.750000,456.750000
50%,8.075415e+05,2.000000,638.500000,480.000000
75%,1.639063e+06,3.000000,999.750000,480.000000
max,2.775638e+06,18.000000,8336.000000,480.000000


In [5]:
embeddings, cache = h.embed_corpus(df, ROOT)
print(embeddings.shape)
display(pd.read_csv(cache/'chunk_diagnostics.csv').describe())
json.loads((cache/'timings.json').read_text())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15969.89it/s]


Embedded 200/7337 articles
Embedded 400/7337 articles
Embedded 600/7337 articles
Embedded 800/7337 articles
Embedded 1000/7337 articles
Embedded 1200/7337 articles
Embedded 1400/7337 articles
Embedded 1600/7337 articles
Embedded 1800/7337 articles
Embedded 2000/7337 articles
Embedded 2200/7337 articles
Embedded 2400/7337 articles
Embedded 2600/7337 articles
Embedded 2800/7337 articles
Embedded 3000/7337 articles
Embedded 3200/7337 articles
Embedded 3400/7337 articles
Embedded 3600/7337 articles
Embedded 3800/7337 articles
Embedded 4000/7337 articles
Embedded 4200/7337 articles
Embedded 4400/7337 articles
Embedded 4600/7337 articles
Embedded 4800/7337 articles
Embedded 5000/7337 articles
Embedded 5200/7337 articles
Embedded 5400/7337 articles
Embedded 5600/7337 articles
Embedded 5800/7337 articles
Embedded 6000/7337 articles
Embedded 6200/7337 articles
Embedded 6400/7337 articles
Embedded 6600/7337 articles
Embedded 6800/7337 articles
Embedded 7000/7337 articles
Embedded 7200/7337 artic

,article_id,chunks,tokens,largest_chunk
count,7.337000e+03,7337.000000,7337.000000,7337.000000
mean,1.084844e+06,2.550361,991.758893,439.988142
std,7.680909e+05,3.305296,1575.783009,73.653148
min,3.000000e+02,1.000000,184.000000,184.000000
25%,4.373710e+05,1.000000,428.000000,428.000000
50%,9.799620e+05,2.000000,647.000000,480.000000
75%,1.638879e+06,3.000000,1071.000000,480.000000
max,2.788848e+06,80.000000,38083.000000,480.000000


{'first_execution': {'model_load_download_seconds': 1.1254650830014725,
  'embedding_seconds': 117.93997074999788,
  'reused_articles': 0,
  'device': 'mps'},
 'latest_execution': {'model_load_download_seconds': 0.10569391700119013,
  'embedding_seconds': 1.4971429580000404,
  'reused_articles': 7337,
  'device': 'mps'}}

## Handoff
Laura should use `Results/modern/corpus.csv` and the ordered IDs in `corpus_manifest.json`. Preserve IDs throughout both pipelines. See `notebooks/README.md` for the common export contract and deadlines.